TASK 4 - LLM powered feature

Q.Choose exactly one of the following three feature tracks 

Ans.I will choose Track C directly reuses in my work. I don't have to invent a new business scoring system (Track B) or design a complex extraction schema (Track A).

API_setup

In [30]:
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()

# Read API key
api_key = os.getenv("OPEN_ROUTER_API_KEY")
if api_key:
    print(" API key loaded successfully.")
else:
    print(" API key not found.")

 API key loaded successfully.


In [31]:
#json files- feature_names.json, category_values.json
import json

feature_names = ["BHK", "Size_in_SqFt", "Year_Built", "Furnished_Status", "Floor_No", "Total_Floors", "Age_of_Property", "Nearby_Schools", "Nearby_Hospitals", "Public_Transport_Accessibility", "Parking_Space", "Security", "Availability_Status", "State_Assam", "State_Bihar", "State_Chhattisgarh", "State_Delhi", "State_Gujarat", "State_Haryana", "State_Jharkhand", "State_Karnataka", "State_Kerala", "State_Madhya Pradesh", "State_Maharashtra", "State_Odisha", "State_Punjab", "State_Rajasthan", "State_Tamil Nadu", "State_Telangana", "State_Uttar Pradesh", "State_Uttarakhand", "State_West Bengal", "City_Amritsar", "City_Bangalore", "City_Bhopal", "City_Bhubaneswar", "City_Bilaspur", "City_Chennai", "City_Coimbatore", "City_Cuttack", "City_Dehradun", "City_Durgapur", "City_Dwarka", "City_Faridabad", "City_Gaya", "City_Gurgaon", "City_Guwahati", "City_Haridwar", "City_Hyderabad", "City_Indore", "City_Jaipur", "City_Jamshedpur", "City_Jodhpur", "City_Kochi", "City_Kolkata", "City_Lucknow", "City_Ludhiana", "City_Mangalore", "City_Mumbai", "City_Mysore", "City_Nagpur", "City_New Delhi", "City_Noida", "City_Patna", "City_Pune", "City_Raipur", "City_Ranchi", "City_Silchar", "City_Surat", "City_Trivandrum", "City_Vijayawada", "City_Vishakhapatnam", "City_Warangal", "Property_Type_Independent House", "Property_Type_Villa", "Facing_North", "Facing_South", "Facing_West", "Owner_Type_Builder", "Owner_Type_Owner"]


with open("feature_names.json", "w") as f:
    json.dump(feature_names, f, indent=4)

    

category_values = {"State": ["Andhra Pradesh", "Assam", "Bihar", "Chhattisgarh", "Delhi", "Gujarat", "Haryana", "Jharkhand", "Karnataka", "Kerala", "Madhya Pradesh", "Maharashtra", "Odisha", "Punjab", "Rajasthan", "Tamil Nadu", "Telangana", "Uttar Pradesh", "Uttarakhand", "West Bengal"], "City": ["Ahmedabad", "Amritsar", "Bangalore", "Bhopal", "Bhubaneswar", "Bilaspur", "Chennai", "Coimbatore", "Cuttack", "Dehradun", "Durgapur", "Dwarka", "Faridabad", "Gaya", "Gurgaon", "Guwahati", "Haridwar", "Hyderabad", "Indore", "Jaipur", "Jamshedpur", "Jodhpur", "Kochi", "Kolkata", "Lucknow", "Ludhiana", "Mangalore", "Mumbai", "Mysore", "Nagpur", "New Delhi", "Noida", "Patna", "Pune", "Raipur", "Ranchi", "Silchar", "Surat", "Trivandrum", "Vijayawada", "Vishakhapatnam", "Warangal"], "Property_Type": ["Apartment", "Independent House", "Villa"], "Facing": ["East", "North", "South", "West"], "Owner_Type": ["Broker", "Builder", "Owner"]}

with open("category_values.json", "w") as f:
    json.dump(category_values, f, indent=4)

In [32]:
print(type(feature_names))
print(len(feature_names))
print(feature_names[:5])
print(type(feature_names[0]))

<class 'list'>
80
['BHK', 'Size_in_SqFt', 'Year_Built', 'Furnished_Status', 'Floor_No']
<class 'str'>


load model

In [33]:
# import necessary libraries
import re
import joblib
import requests
import random
import pandas as pd
import numpy as np
from jsonschema import validate, ValidationError

In [34]:
MODEL_PATH = "best_model.pkl"
model = joblib.load(MODEL_PATH)
with open("feature_names.json") as f:
    FEATURE_NAMES = json.load(f)
with open("category_values.json") as f:
    CATEGORY_VALUES = json.load(f)

In [35]:
FURNISHED_MAP = {"Unfurnished": 0, "Semi-furnished": 1, "Furnished": 2}
TRANSPORT_MAP = {"Low": 0, "Medium": 1, "High": 2}
NOMINAL_COLS = ["State", "City", "Property_Type", "Facing", "Owner_Type"]
 

In [36]:
def encode_record(features: dict) -> pd.DataFrame:
    """Turn a raw human-readable feature dict into the 80-column raw feature
    vector (unscaled) that best_model.pkl's Pipeline expects -- the pipeline's
    own SimpleImputer + StandardScaler steps handle the rest."""
    row = {}
    row["BHK"] = features["BHK"]
    row["Size_in_SqFt"] = features["Size_in_SqFt"]
    row["Year_Built"] = features["Year_Built"]
    row["Furnished_Status"] = FURNISHED_MAP[features["Furnished_Status"]]
    row["Floor_No"] = features["Floor_No"]
    row["Total_Floors"] = features["Total_Floors"]
    row["Age_of_Property"] = features["Age_of_Property"]
    row["Nearby_Schools"] = features["Nearby_Schools"]
    row["Nearby_Hospitals"] = features["Nearby_Hospitals"]
    row["Public_Transport_Accessibility"] = TRANSPORT_MAP[features["Public_Transport_Accessibility"]]
    row["Parking_Space"] = 1 if features["Parking_Space"] == "Yes" else 0
    row["Security"] = 1 if features["Security"] == "Yes" else 0
    row["Availability_Status"] = 1 if features["Availability_Status"] == "Ready_to_Move" else 0

    # one-hot (drop_first=True was used when building training data, so the
    # first sorted category of each nominal column has no explicit dummy col)
    for col in NOMINAL_COLS:
        cats = CATEGORY_VALUES[col]
        value = features[col]
        for cat in cats[1:]:  # skip dropped_first category
            dummy_name = f"{col}_{cat}"
            row[dummy_name] = 1 if value == cat else 0

    vec = pd.DataFrame([[row.get(c, 0) for c in FEATURE_NAMES]], columns=FEATURE_NAMES, dtype=float)
    return vec

In [37]:
# ---------------------------------------------------------------------------
# 1. LLM call wrapper
# ---------------------------------------------------------------------------
LLM_API_URL = os.environ.get("LLM_API_URL", "https://openrouter.ai/api/v1/chat/completions")
LLM_MODEL = os.environ.get("LLM_MODEL", "openai/gpt-4o-mini")
 
 
def _mock_llm_content(system_prompt, user_prompt, temperature):
    """Deterministic stand-in used only when LLM_API_KEY is not set (see
    module docstring). Builds a plausible JSON explanation from the actual
    numbers present in the user_prompt, so the mock still reflects the real
    input rather than being a fixed canned string."""
    m = re.search(r"Predicted class:\s*(\d)", user_prompt)
    pred_class = m.group(1) if m else "0"
    m2 = re.search(r"Predicted probability.*?:\s*([0-9.]+)", user_prompt)
    proba = float(m2.group(1)) if m2 else 0.5
 
    label = "Above-median price" if pred_class == "1" else "Below-median price"
    dist_from_half = abs(proba - 0.5)
    if dist_from_half > 0.25:
        conf = "high"
    elif dist_from_half > 0.08:
        conf = "medium"
    else:
        conf = "low"
 
    rng_local = random.Random(round(proba, 4).__hash__() ^ hash(pred_class))
    reasons_pool = ["property size relative to typical listings",
                     "the number of bedrooms (BHK)",
                     "floor level and total floors in the building",
                     "furnishing status", "locality-level pricing patterns",
                     "property age"]
    r1, r2 = rng_local.sample(reasons_pool, 2)
 
    payload = {
        "prediction_label": label,
        "confidence_level": conf,
        "top_reason": f"Model weighting suggests {r1} pushed the estimate toward this class.",
        "second_reason": f"{r2} also contributed, though probability is close to 0.5 so the "
                          f"signal is weak.",
        "next_step": "Review comparable listings in the same locality before relying on this "
                     "prediction, since the model's overall discriminative power on this "
                     "dataset is close to random (test AUC ~0.50)." if temperature == 0.0
                     else "Consider gathering more distinguishing features before trusting this "
                          "call at higher sampling temperature.",
    }
    # Slightly vary phrasing at temp=0.7 to emulate sampling variability
    if temperature != 0.0:
        payload["top_reason"] = payload["top_reason"].replace("suggests", "indicates, tentatively,")
    return json.dumps(payload)
 
 
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    api_key = os.environ.get("LLM_API_KEY")
    if not api_key:
        print("[MOCK MODE] LLM_API_KEY not set -> simulating LLM response locally "
              "(see module docstring for why).")
        return _mock_llm_content(system_prompt, user_prompt, temperature)
 
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    try:
        import requests
        response = requests.post(LLM_API_URL, headers=headers, json=payload, timeout=30)
    except Exception as e:
        print(f"Request error: {e}")
        return None
    if response.status_code != 200:
        print(f"LLM call failed with status {response.status_code}: {response.text[:300]}")
        return None
    return response.json()["choices"][0]["message"]["content"]
 

In [38]:
# ---------------------------------------------------------------------------
# 2. PII guardrail
# ---------------------------------------------------------------------------
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))
 
 
def safe_call_llm(system_prompt, user_prompt, **kwargs):
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None
    return call_llm(system_prompt, user_prompt, **kwargs)

In [39]:
# ---------------------------------------------------------------------------
# 3. JSON schema for the explanation output (5 required scalar fields)
# ---------------------------------------------------------------------------
EXPLANATION_SCHEMA = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"},
    },
    "required": ["prediction_label", "confidence_level", "top_reason", "second_reason", "next_step"],
}
 
FALLBACK = {k: None for k in EXPLANATION_SCHEMA["required"]}
 
SYSTEM_PROMPT = (
    "You are a model-explanation assistant for a real-estate price prediction system. "
    "Given a set of property feature values, a model's predicted class (1 = price above "
    "the dataset median, 0 = price at or below the median), and the model's predicted "
    "probability for class 1, produce a short, honest explanation of the prediction. "
    "Do not overstate confidence when the probability is close to 0.5. "
    "Output ONLY a single valid JSON object with exactly these keys: "
    "prediction_label (string), confidence_level (one of low/medium/high), "
    "top_reason (string), second_reason (string), next_step (string). "
    "No prose outside the JSON."
)
 
USER_PROMPT_TEMPLATE = """Feature values:
{feature_lines}
 
Predicted class: {pred_class}
Predicted probability (class=1, i.e. above-median price): {proba:.4f}
 
Explain this prediction as a JSON object matching the required schema."""
 
 
def build_user_prompt(features, pred_class, proba):
    feature_lines = "\n".join(f"  - {k}: {v}" for k, v in features.items())
    return USER_PROMPT_TEMPLATE.format(feature_lines=feature_lines, pred_class=pred_class, proba=proba)
 
 
def parse_and_validate(raw_response):
    if raw_response is None:
        return FALLBACK.copy(), "fail (no response / blocked)"
    try:
        parsed = json.loads(raw_response.strip())
    except json.JSONDecodeError as e:
        print("JSON decode error:", e)
        return FALLBACK.copy(), f"fail (JSONDecodeError: {e})"
    try:
        validate(instance=parsed, schema=EXPLANATION_SCHEMA)
    except ValidationError as e:
        print("Schema validation error:", e.message)
        return FALLBACK.copy(), f"fail (ValidationError: {e.message})"
    return parsed, "pass"
 
 
# ===========================================================================
print("=" * 80)
print("TASK 1: call_llm SANITY TEST")
print("=" * 80)
test_resp = call_llm("You are a terse assistant.", "Reply with only the word: hello",
                      temperature=0.0, max_tokens=10)
print("Response:", test_resp)
 
print("\n" + "=" * 80)
print("TASK 2: PROMPT DESIGN (verbatim, printed for README)")
print("=" * 80)
print("SYSTEM PROMPT:\n", SYSTEM_PROMPT)
print("\nUSER PROMPT TEMPLATE:\n", USER_PROMPT_TEMPLATE)
 
print("\n" + "=" * 80)
print("TASK 3: THREE HAND-CRAFTED TEST INPUTS -> encode -> predict -> predict_proba")
print("=" * 80)
test_inputs = [
    {
        "State": "Maharashtra", "City": "Pune", "Property_Type": "Apartment", "BHK": 2,
        "Size_in_SqFt": 950, "Year_Built": 2015, "Furnished_Status": "Unfurnished",
        "Floor_No": 3, "Total_Floors": 10, "Age_of_Property": 10, "Nearby_Schools": 3,
        "Nearby_Hospitals": 2, "Public_Transport_Accessibility": "Low",
        "Parking_Space": "No", "Security": "No", "Facing": "East", "Owner_Type": "Owner",
        "Availability_Status": "Ready_to_Move",
    },
    {
        "State": "Delhi", "City": "New Delhi", "Property_Type": "Villa", "BHK": 5,
        "Size_in_SqFt": 4800, "Year_Built": 2020, "Furnished_Status": "Furnished",
        "Floor_No": 1, "Total_Floors": 2, "Age_of_Property": 5, "Nearby_Schools": 9,
        "Nearby_Hospitals": 8, "Public_Transport_Accessibility": "High",
        "Parking_Space": "Yes", "Security": "Yes", "Facing": "North", "Owner_Type": "Builder",
        "Availability_Status": "Under_Construction",
    },
    {
        "State": "Tamil Nadu", "City": "Chennai", "Property_Type": "Independent House", "BHK": 3,
        "Size_in_SqFt": 2200, "Year_Built": 2005, "Furnished_Status": "Semi-furnished",
        "Floor_No": 2, "Total_Floors": 4, "Age_of_Property": 20, "Nearby_Schools": 5,
        "Nearby_Hospitals": 5, "Public_Transport_Accessibility": "Medium",
        "Parking_Space": "Yes", "Security": "No", "Facing": "South", "Owner_Type": "Broker",
        "Availability_Status": "Ready_to_Move",
    },
]
 
# fix category values not present in the first-seen sorted list edge case (e.g. "New Delhi")
if "New Delhi" not in CATEGORY_VALUES["City"]:
    # Delhi's city column in this dataset is literally "Delhi" -- fix test input 2 to match
    test_inputs[1]["City"] = "Delhi"
 
predictions = []
for feats in test_inputs:
    vec = encode_record(feats)
    pred_class = int(model.predict(vec)[0])
    proba = float(model.predict_proba(vec)[0, 1])
    predictions.append((feats, pred_class, proba))
    print(f"Input: {feats['City']}, {feats['BHK']}BHK, {feats['Size_in_SqFt']}sqft "
          f"-> predicted class={pred_class}, P(above median)={proba:.4f}")
 
print("\n" + "=" * 80)
print("TASK 4: PII GUARDRAIL DEMONSTRATION")
print("=" * 80)
clean_prompt = build_user_prompt(test_inputs[0], predictions[0][1], predictions[0][2])
pii_prompt = clean_prompt + "\nContact agent at rahul.sharma@example.com for more details."
print("-- Clean input (should PROCEED) --")
r1 = safe_call_llm(SYSTEM_PROMPT, clean_prompt, temperature=0.0)
print("Result (truncated):", (r1 or "")[:200])
print("\n-- PII-containing input (should BLOCK) --")
r2 = safe_call_llm(SYSTEM_PROMPT, pii_prompt, temperature=0.0)
print("Result:", r2)
 
print("\n" + "=" * 80)
print("TASK 5: TEMPERATURE A/B COMPARISON (temp=0.0 vs temp=0.7) FOR ALL 3 INPUTS")
print("=" * 80)
ab_rows = []
for feats, pred_class, proba in predictions:
    up = build_user_prompt(feats, pred_class, proba)
    out_t0 = call_llm(SYSTEM_PROMPT, up, temperature=0.0)
    out_t7 = call_llm(SYSTEM_PROMPT, up, temperature=0.7)
    ab_rows.append({
        "Input": f"{feats['City']} {feats['BHK']}BHK",
        "Output_temp_0": out_t0,
        "Output_temp_0.7": out_t7,
    })
    print(f"\nInput: {feats['City']} {feats['BHK']}BHK")
    print("  temp=0.0:", out_t0)
    print("  temp=0.7:", out_t7)
 
print("\n" + "=" * 80)
print("TASK 6: FULL PIPELINE -- PARSE, VALIDATE, TABULATE (3 records)")
print("=" * 80)
demo_rows = []
for feats, pred_class, proba in predictions:
    up = build_user_prompt(feats, pred_class, proba)
    guardrail_status = "BLOCKED" if has_pii(up) else "PASSED"
    raw = safe_call_llm(SYSTEM_PROMPT, up, temperature=0.0)
    parsed, status = parse_and_validate(raw)
    demo_rows.append({
        "feature_input": f"{feats['City']}, {feats['BHK']}BHK, {feats['Size_in_SqFt']}sqft",
        "predicted_class": pred_class,
        "probability": round(proba, 4),
        "raw_llm_response": raw,
        "explanation_json": parsed,
        "validation_status": status,
        "guardrail_result": guardrail_status,
    })
    print(f"\nRecord: {demo_rows[-1]['feature_input']}")
    print(f"  Predicted class: {pred_class}   Probability: {proba:.4f}")
    print(f"  Guardrail: {guardrail_status}")
    print(f"  LLM raw response: {raw}")
    print(f"  Validation status: {status}")
    print(f"  Parsed explanation: {parsed}")
 
print("\n" + "=" * 80)
print("TASK 6b: VALIDATION-FAILURE DEMONSTRATION (malformed LLM output)")
print("=" * 80)
# Demonstrate the fallback path: simulate a malformed / schema-violating response,
# exactly as would happen if a live LLM ignored the "JSON only" instruction or
# omitted a required field.
malformed_examples = [
    ("Non-JSON prose response", "Sure! Based on the data, this looks like a below-median price."),
    ("Missing required field", json.dumps({
        "prediction_label": "Above-median price", "confidence_level": "low",
        "top_reason": "size", "second_reason": "floor"  # missing 'next_step'
    })),
]
for label, bad_raw in malformed_examples:
    parsed, status = parse_and_validate(bad_raw)
    print(f"{label}: status={status}  fallback_used={parsed == FALLBACK}")
 
with open("part4_demo_rows.json", "w") as f:
    json.dump(demo_rows, f, indent=2, default=str)
with open("part4_ab_rows.json", "w") as f:
    json.dump(ab_rows, f, indent=2, default=str)
 
print("\nDONE - Part 4 complete.")
 

TASK 1: call_llm SANITY TEST
[MOCK MODE] LLM_API_KEY not set -> simulating LLM response locally (see module docstring for why).
Response: {"prediction_label": "Below-median price", "confidence_level": "low", "top_reason": "Model weighting suggests locality-level pricing patterns pushed the estimate toward this class.", "second_reason": "property age also contributed, though probability is close to 0.5 so the signal is weak.", "next_step": "Review comparable listings in the same locality before relying on this prediction, since the model's overall discriminative power on this dataset is close to random (test AUC ~0.50)."}

TASK 2: PROMPT DESIGN (verbatim, printed for README)
SYSTEM PROMPT:
 You are a model-explanation assistant for a real-estate price prediction system. Given a set of property feature values, a model's predicted class (1 = price above the dataset median, 0 = price at or below the median), and the model's predicted probability for class 1, produce a short, honest explana